- The classic optical SNR cut is `[SII]/Halpha > 0.4`, because radiative shocks enhance low-ionization `[SII]` compared with HII regions.
- The PHANGS-MUSE SNR paper (Li et al. 2024, A&A 690, A161) shows why this alone is not enough: DIG also has elevated `[SII]/Halpha`, especially at low Halpha surface brightness.
- Their solution is to fit the normal photoionized/DIG trend in `log([SII]/Halpha)` versus `log(Halpha surface brightness)`, subtract it, and run `astrodendro` on the positive residual map. They classify robust SNRs when more than one diagnostic agrees, or when integrated shock-line ratios are high.
- The M33 paper appendix summarizes the Local Group style selection from Lee & Lee (2014): `[SII]/Halpha > 0.4`, round/shell morphology, and no blue stars in the centre. We can automate the first part and flag morphology for visual checking.

In [5]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi

from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u

from astrodendro import Dendrogram
from skimage.measure import regionprops
from skimage.morphology import binary_dilation, disk

from tqdm.notebook import tqdm

BASE = Path('.')
FITS_PATH = BASE / 'ADP.2025-07-06T07_52_11.876.fits'
PN_CAT_PATH = BASE / 'PN_CAT.csv'
HII_CAT_PATH = BASE / 'HII_dendro_ipm.csv'
HII_LABEL_PATH = BASE / 'HII_labels_refined.fits'

pix_scale = 0.2
psf_fwhm_arcsec = 0.85
psf_fwhm_pix = psf_fwhm_arcsec / pix_scale

In [2]:
def clean_map(x):
    y = np.array(x, dtype=float, copy=True)
    y[~np.isfinite(y)] = 0.0
    y[y < 0] = 0.0
    return y


def safe_ratio(num, den):
    out = np.full_like(num, np.nan, dtype=float)
    good = np.isfinite(num) & np.isfinite(den) & (num > 0) & (den > 0)
    out[good] = num[good] / den[good]
    return out


def safe_log10(x):
    out = np.full_like(x, np.nan, dtype=float)
    good = np.isfinite(x) & (x > 0)
    out[good] = np.log10(x[good])
    return out


def sum_in_mask(arr, mask):
    x = np.asarray(arr, dtype=float)
    return np.nansum(x[mask & np.isfinite(x)])


def err_sum_in_mask(err, mask):
    e = np.asarray(err, dtype=float)
    good = mask & np.isfinite(e) & (e > 0)
    return np.sqrt(np.nansum(e[good]**2))


def robust_sigma(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    return 1.4826 * mad


def percentile_norm(img, pmin=1, pmax=99):
    x = np.array(img, dtype=float, copy=True)
    x[~np.isfinite(x)] = np.nan
    lo, hi = np.nanpercentile(x, [pmin, pmax])
    return np.clip((x - lo) / (hi - lo), 0, 1)

In [3]:
hdul = fits.open(FITS_PATH, memmap=True)

ha = hdul['HA6562_FLUX'].data
hb = hdul['HB4861_FLUX'].data
oiii = hdul['OIII5006_FLUX'].data
nii = hdul['NII6583_FLUX'].data
sii6716 = hdul['SII6716_FLUX'].data
sii6730 = hdul['SII6730_FLUX'].data
sii = sii6716 + sii6730

ha_err = hdul['HA6562_FLUX_ERR'].data
hb_err = hdul['HB4861_FLUX_ERR'].data
oiii_err = hdul['OIII5006_FLUX_ERR'].data
nii_err = hdul['NII6583_FLUX_ERR'].data
sii6716_err = hdul['SII6716_FLUX_ERR'].data
sii6730_err = hdul['SII6730_FLUX_ERR'].data
sii_err = np.sqrt(sii6716_err**2 + sii6730_err**2)

wcs = WCS(hdul['HA6562_FLUX'].header)
shape = ha.shape
print('Map shape:', shape)

Map shape: (1733, 6080)


In [6]:
def mask_pn_cat(cat, wcs, shape, radius_factor=2.5, min_radius_pix=None):
    mask = np.zeros(shape, dtype=bool)
    if len(cat) == 0:
        return mask

    coords = SkyCoord(cat['RAdeg'].values * u.deg, cat['DEdeg'].values * u.deg)
    x, y = wcs.world_to_pixel(coords)
    yy, xx = np.indices(shape)

    if min_radius_pix is None:
        min_radius_pix = psf_fwhm_pix

    for xi, yi, fwhm_arcsec in zip(x, y, cat['FWHM'].values):
        if not np.isfinite(xi) or not np.isfinite(yi):
            continue
        r_pix = max(radius_factor * fwhm_arcsec / pix_scale, min_radius_pix)
        mask |= (xx - xi)**2 + (yy - yi)**2 <= r_pix**2

    return mask


def hii_rows_to_mask(hii_cat, wcs, shape, radius_factor=1.35, min_radius_pix=None):
    """Fallback mask if the HII segmentation FITS was not saved.

    This uses circular masks with area scaled from npix. It is deliberately
    conservative because the request is to mask confident and possible HII.
    """
    mask = np.zeros(shape, dtype=bool)
    if len(hii_cat) == 0:
        return mask

    if min_radius_pix is None:
        min_radius_pix = psf_fwhm_pix

    yy, xx = np.indices(shape)
    for row in hii_cat.itertuples(index=False):
        xi = getattr(row, 'x')
        yi = getattr(row, 'y')
        npix = getattr(row, 'npix')
        if not (np.isfinite(xi) and np.isfinite(yi) and np.isfinite(npix)):
            continue
        r_pix = max(radius_factor * np.sqrt(npix / np.pi), min_radius_pix)
        mask |= (xx - xi)**2 + (yy - yi)**2 <= r_pix**2
    return mask


def select_hii_to_mask(hii_cat):
    """Mask confident and possible HII, but keep already-flagged SNR-like rows out."""
    hii_cat = hii_cat.copy()

    if {'clean_hii', 'possible_hii'}.issubset(hii_cat.columns):
        use = hii_cat['clean_hii'].astype(bool) | hii_cat['possible_hii'].astype(bool)
    elif 'simple_class' in hii_cat.columns:
        use = hii_cat['simple_class'].isin(['HII', 'HII_clean', 'HII_possible', 'possible_hii'])
    elif 'bpt_class' in hii_cat.columns:
        use = hii_cat['bpt_class'].isin(['HII_BPT_clean', 'HII_BPT_possible'])
        if 'snr_like' in hii_cat.columns:
            use &= ~hii_cat['snr_like'].astype(bool)
    else:
        use = np.ones(len(hii_cat), dtype=bool)

    return hii_cat.loc[use].copy()


pn_cat = pd.read_csv(PN_CAT_PATH)
hii_cat = pd.read_csv(HII_CAT_PATH)
hii_to_mask = select_hii_to_mask(hii_cat)

pn_mask = mask_pn_cat(pn_cat, wcs, shape)

if HII_LABEL_PATH.exists():
    hii_labels = fits.getdata(HII_LABEL_PATH).astype(np.int32)
    masked_ids = set(hii_to_mask['region_id'].astype(int))
    hii_mask = np.isin(hii_labels, list(masked_ids))
    mask_source = HII_LABEL_PATH.name
else:
    hii_mask = hii_rows_to_mask(hii_to_mask, wcs, shape)
    mask_source = 'catalogue circular fallback'

exclude_mask = pn_mask | hii_mask
print(f'PNs masked: {pn_mask.sum():,} pixels')
print(f'HII catalogue: {hii_cat_path.name} ({len(hii_cat)} rows)')
print(f'HII rows masked: {len(hii_to_mask)} using {mask_source}')
print(f'Total excluded: {exclude_mask.sum():,} pixels')

PNs masked: 185,114 pixels


NameError: name 'hii_cat_path' is not defined

## Build SNR diagnostic maps

The paper's key idea is not simply to select high `[SII]/Halpha`. It first models the normal photoionized/DIG background relation between `log([SII]/Halpha)` and `log(Halpha surface brightness)`. SNR candidates are then compact positive residuals above that relation.

Because we already mask PNe and HII regions, the fitted relation below is mostly the remaining diffuse/background emission. The classic `[SII]/Halpha >= 0.4` flag is still kept as an independent diagnostic.

In [ ]:
min_line_snr = 5.0
classic_sii_ha_thresh = 0.40
possible_sii_ha_thresh = 0.35
residual_sigma_thresh = 2.0
residual_min_delta = 0.10       # dex, matching the PHANGS-MUSE choice
min_npix = 8                    # tune after visual inspection
max_npix = 5000                 # prevents huge DIG sheets from entering the compact-source catalogue

sn_ha_map = np.full(shape, np.nan, dtype=float)
sn_sii_map = np.full(shape, np.nan, dtype=float)
good_ha_err = np.isfinite(ha_err) & (ha_err > 0)
good_sii_err = np.isfinite(sii_err) & (sii_err > 0)
sn_ha_map[good_ha_err] = ha[good_ha_err] / ha_err[good_ha_err]
sn_sii_map[good_sii_err] = sii[good_sii_err] / sii_err[good_sii_err]

sii_ha_map = safe_ratio(sii, ha)
log_sii_ha_map = safe_log10(sii_ha_map)
log_ha_surface = safe_log10(ha)   # per-pixel Halpha surface brightness; constant pixel area only shifts intercept

valid_ratio = (
    np.isfinite(log_sii_ha_map)
    & np.isfinite(log_ha_surface)
    & (sn_ha_map >= min_line_snr)
    & (sn_sii_map >= min_line_snr)
    & (~exclude_mask)
)

# Fit the dominant photoionized/DIG trend. Iterative clipping keeps obvious shock outliers from driving the fit.
fit_mask = valid_ratio.copy()
for _ in range(4):
    x = log_ha_surface[fit_mask]
    y = log_sii_ha_map[fit_mask]
    slope, intercept = np.polyfit(x, y, deg=1)
    resid = y - (slope * x + intercept)
    sig = robust_sigma(resid)
    fit_mask[fit_mask] = np.abs(resid) < 3 * sig

log_sii_ha_fit = slope * log_ha_surface + intercept
sii_ha_residual = log_sii_ha_map - log_sii_ha_fit
resid_sigma = robust_sigma(sii_ha_residual[fit_mask])

print(f'Fitted log([SII]/Halpha) = {slope:.3f} log(Halpha) + {intercept:.3f}')
print(f'Residual sigma = {resid_sigma:.3f} dex')
print(f'Valid unmasked pixels for detection: {valid_ratio.sum():,}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
idx = np.where(valid_ratio)
if len(idx[0]) > 200_000:
    rng = np.random.default_rng(42)
    take = rng.choice(len(idx[0]), size=200_000, replace=False)
    yy, xx = idx[0][take], idx[1][take]
else:
    yy, xx = idx

ax.scatter(log_ha_surface[yy, xx], log_sii_ha_map[yy, xx], s=1, alpha=0.08, color='0.2')
xgrid = np.linspace(np.nanpercentile(log_ha_surface[valid_ratio], 1), np.nanpercentile(log_ha_surface[valid_ratio], 99), 100)
ax.plot(xgrid, slope * xgrid + intercept, color='crimson', lw=2, label='photoionized/DIG fit')
ax.axhline(np.log10(classic_sii_ha_thresh), color='navy', ls='--', lw=1.5, label='[SII]/Halpha = 0.4')
ax.set_xlabel('log Halpha surface brightness')
ax.set_ylabel('log([SII]/Halpha)')
ax.legend()
plt.tight_layout()
plt.show()

## Detect compact SNR candidates

Two masks are used:

- `residual_seed`: pixels above the fitted photoionized/DIG relation by at least `2 sigma`, following the PHANGS-MUSE spirit.
- `classic_seed`: pixels with `[SII]/Halpha >= 0.4`, the classic Local Group/M33 criterion.

The final candidate objects are astrodendro leaves in the residual map plus any compact classic-threshold islands not already recovered. The catalogue records which diagnostic selected each region.

In [ ]:
residual_detection = np.array(sii_ha_residual, dtype=float, copy=True)
residual_detection[~valid_ratio] = np.nan
residual_detection[residual_detection < 0] = 0.0

residual_seed = valid_ratio & (sii_ha_residual >= residual_sigma_thresh * resid_sigma)
classic_seed = valid_ratio & (sii_ha_map >= classic_sii_ha_thresh)
possible_seed = valid_ratio & (sii_ha_map >= possible_sii_ha_thresh)

# Astrodendro cannot use NaN as background.
dendro_input = np.nan_to_num(residual_detection, nan=0.0, posinf=0.0, neginf=0.0)
d = Dendrogram.compute(
    dendro_input,
    min_value=residual_sigma_thresh * resid_sigma,
    min_delta=residual_min_delta,
    min_npix=min_npix,
)

snr_labels = np.zeros(shape, dtype=np.int32)
method_by_id = {}
next_id = 1

for leaf in d.leaves:
    leaf_mask = leaf.get_mask() & valid_ratio
    npix = int(leaf_mask.sum())
    if npix < min_npix or npix > max_npix:
        continue
    snr_labels[leaf_mask] = next_id
    method_by_id[next_id] = {'residual'}
    next_id += 1

classic_islands, n_islands = ndi.label(classic_seed & (snr_labels == 0))
for island_id in range(1, n_islands + 1):
    island = classic_islands == island_id
    npix = int(island.sum())
    if npix < min_npix or npix > max_npix:
        continue
    snr_labels[island] = next_id
    method_by_id[next_id] = {'classic'}
    next_id += 1

# Add method flags for all labels based on overlap.
for rid in range(1, next_id):
    mask = snr_labels == rid
    if np.any(classic_seed & mask):
        method_by_id[rid].add('classic')
    if np.any(possible_seed & mask):
        method_by_id[rid].add('possible_0p35')

print(f'Residual dendrogram leaves kept: {sum("residual" in m for m in method_by_id.values())}')
print(f'Classic-only islands added: {sum(m == {"classic", "possible_0p35"} or m == {"classic"} for m in method_by_id.values())}')
print(f'Total SNR candidate regions: {len(method_by_id)}')

In [ ]:
def make_snr_catalog(labels):
    rows = []
    props = {p.label: p for p in regionprops(labels)}

    for rid in sorted(method_by_id):
        mask = labels == rid
        npix = int(mask.sum())
        if npix == 0:
            continue

        F_ha = sum_in_mask(ha, mask)
        F_hb = sum_in_mask(hb, mask)
        F_oiii = sum_in_mask(oiii, mask)
        F_nii = sum_in_mask(nii, mask)
        F_sii = sum_in_mask(sii, mask)

        E_ha = err_sum_in_mask(ha_err, mask)
        E_hb = err_sum_in_mask(hb_err, mask)
        E_oiii = err_sum_in_mask(oiii_err, mask)
        E_nii = err_sum_in_mask(nii_err, mask)
        E_sii = err_sum_in_mask(sii_err, mask)

        sn_ha = F_ha / E_ha if E_ha > 0 else np.nan
        sn_hb = F_hb / E_hb if E_hb > 0 else np.nan
        sn_oiii = F_oiii / E_oiii if E_oiii > 0 else np.nan
        sn_nii = F_nii / E_nii if E_nii > 0 else np.nan
        sn_sii = F_sii / E_sii if E_sii > 0 else np.nan

        sii_ha = F_sii / F_ha if F_ha > 0 and F_sii > 0 else np.nan
        nii_ha = F_nii / F_ha if F_ha > 0 and F_nii > 0 else np.nan
        oiii_hb = F_oiii / F_hb if F_hb > 0 and F_oiii > 0 else np.nan

        yy, xx = np.where(mask)
        xcen = float(np.mean(xx))
        ycen = float(np.mean(yy))
        ra, dec = wcs.wcs_pix2world(xcen, ycen, 0)

        prop = props.get(rid)
        eccentricity = prop.eccentricity if prop is not None else np.nan
        area = prop.area if prop is not None else np.nan
        equiv_radius_pix = np.sqrt(area / np.pi) if np.isfinite(area) else np.nan

        methods = method_by_id[rid]
        selected_residual = 'residual' in methods
        selected_classic = 'classic' in methods
        integrated_classic = np.isfinite(sii_ha) and sii_ha >= classic_sii_ha_thresh
        high_residual = np.nanmax(sii_ha_residual[mask]) >= residual_sigma_thresh * resid_sigma

        if selected_residual and integrated_classic:
            snr_class = 'SNR_high_confidence'
        elif integrated_classic or high_residual:
            snr_class = 'SNR_candidate'
        else:
            snr_class = 'SNR_low_priority'

        rows.append({
            'snr_id': rid,
            'npix': npix,
            'x': xcen,
            'y': ycen,
            'ra': ra,
            'dec': dec,
            'equiv_radius_pix': equiv_radius_pix,
            'equiv_radius_arcsec': equiv_radius_pix * pix_scale,
            'eccentricity': eccentricity,
            'selection_methods': '+'.join(sorted(methods)),
            'selected_residual': selected_residual,
            'selected_classic_sii_ha': selected_classic,
            'F_ha': F_ha,
            'F_hb': F_hb,
            'F_oiii': F_oiii,
            'F_nii': F_nii,
            'F_sii': F_sii,
            'sn_ha': sn_ha,
            'sn_hb': sn_hb,
            'sn_oiii': sn_oiii,
            'sn_nii': sn_nii,
            'sn_sii': sn_sii,
            'sii_ha': sii_ha,
            'nii_ha': nii_ha,
            'oiii_hb': oiii_hb,
            'log_sii_ha': np.log10(sii_ha) if np.isfinite(sii_ha) else np.nan,
            'log_nii_ha': np.log10(nii_ha) if np.isfinite(nii_ha) else np.nan,
            'log_oiii_hb': np.log10(oiii_hb) if np.isfinite(oiii_hb) else np.nan,
            'ha_surface': F_ha / npix if npix > 0 else np.nan,
            'residual_median': np.nanmedian(sii_ha_residual[mask]),
            'residual_max': np.nanmax(sii_ha_residual[mask]),
            'snr_class': snr_class,
            'needs_visual_shell_check': True,
        })

    return pd.DataFrame(rows).sort_values(['snr_class', 'sii_ha', 'residual_max'], ascending=[True, False, False])

snr_cat = make_snr_catalog(snr_labels)
snr_cat.to_csv('SNR_candidates.csv', index=False)
fits.writeto('SNR_candidate_labels.fits', snr_labels.astype(np.int32), hdul['HA6562_FLUX'].header, overwrite=True)

print(snr_cat['snr_class'].value_counts(dropna=False))
snr_cat.head(20)

In [ ]:
candidate_footprint = snr_labels > 0

fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)

axes[0].imshow(percentile_norm(ha), origin='lower', cmap='gray')
axes[0].imshow(np.ma.masked_where(~exclude_mask, exclude_mask), origin='lower', cmap='Reds', alpha=0.35)
axes[0].set_title('Masked PNe + HII')

im1 = axes[1].imshow(sii_ha_residual, origin='lower', cmap='magma', vmin=0, vmax=np.nanpercentile(sii_ha_residual[valid_ratio], 99))
if candidate_footprint.any():
    axes[1].contour(candidate_footprint.astype(float), levels=[0.5], colors='cyan', linewidths=0.5)
axes[1].set_title('[SII]/Halpha residual + candidates')
fig.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(log_sii_ha_map, origin='lower', cmap='viridis', vmin=-1.2, vmax=0.1)
if candidate_footprint.any():
    axes[2].contour(candidate_footprint.astype(float), levels=[0.5], colors='white', linewidths=0.5)
axes[2].set_title('log([SII]/Halpha) + candidates')
fig.colorbar(im2, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.show()

In [ ]:
def plot_snr_cutouts(cat, labels, image=None, n=12, pad=20, sort_by='residual_max'):
    if image is None:
        image = sii_ha_residual

    sample = cat.sort_values(sort_by, ascending=False).head(n)
    for row in sample.itertuples(index=False):
        rid = int(row.snr_id)
        mask = labels == rid
        yy, xx = np.where(mask)
        if len(xx) == 0:
            continue
        y0 = max(yy.min() - pad, 0)
        y1 = min(yy.max() + pad + 1, labels.shape[0])
        x0 = max(xx.min() - pad, 0)
        x1 = min(xx.max() + pad + 1, labels.shape[1])

        img_cut = image[y0:y1, x0:x1]
        mask_cut = mask[y0:y1, x0:x1]
        ha_cut = ha[y0:y1, x0:x1]

        fig, axes = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)
        vmin, vmax = np.nanpercentile(img_cut[np.isfinite(img_cut)], [2, 98])
        axes[0].imshow(img_cut, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
        axes[0].contour(mask_cut.astype(float), levels=[0.5], colors='cyan', linewidths=1.5)
        axes[0].set_title(f'SNR {rid}: residual')

        axes[1].imshow(percentile_norm(ha_cut), origin='lower', cmap='gray')
        axes[1].contour(mask_cut.astype(float), levels=[0.5], colors='red', linewidths=1.5)
        axes[1].set_title(f"{row.snr_class}\n[SII]/Ha={row.sii_ha:.2f}, max res={row.residual_max:.2f}")

        for ax in axes:
            ax.set_xticks([])
            ax.set_yticks([])
        plt.show()

plot_snr_cutouts(snr_cat, snr_labels, n=12)

## Optional improvement for the HII notebook

If `HII_labels_refined.fits` does not exist, add this at the end of `astrodendro.ipynb` after `labels_refined` is created:

```python
fits.writeto('HII_labels_refined.fits', labels_refined.astype(np.int32), header, overwrite=True)
```

That makes the HII mask exact. The circular fallback above is conservative, but exact labels are better for a thesis-quality catalogue.